# M1 고정 CLV 및 N/V 구성별 신규상품 오차 진단 v1.4

기존 Dunnhumby seed 42 M1 체크포인트를 **재학습하지 않고** 분석합니다.

기존 저·중·고 CLV 분석에 전체 N/V 4유형과 고CLV 내부 V우세·균형·N우세 분석을 추가합니다. 모든 경계는 학습이력으로만 고정하며, 성과를 보고 바꾸지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '2af26e19ab1066e3752eb92633caff9a83d35f9f'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA

import subprocess
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
print('진단 코드 고정:', actual_sha)

In [ ]:
import importlib
import json
import torch
import lightgcn_clv_fixed_segment_error_diagnostic as diagnostic_module

diagnostic_module = importlib.reload(diagnostic_module)
assert diagnostic_module.CODE_VERSION == 'm1-fixed-clv-segment-error-diagnostic-v1.4', (
    diagnostic_module.__file__, diagnostic_module.CODE_VERSION
)
configure_fixed_segment_error_diagnostic = diagnostic_module.configure_fixed_segment_error_diagnostic
preflight_summary = diagnostic_module.preflight_summary
run_fixed_segment_error_diagnostic = diagnostic_module.run_fixed_segment_error_diagnostic

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_fixed_segment_error_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m1_fixed_clv_segment_error_diagnostic_v14'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    eval_batch_size=32,
    top_examples=20,
)
summary = preflight_summary(cfg)
assert summary['code_version'] == 'm1-fixed-clv-segment-error-diagnostic-v1.4', summary
assert summary['segments'] == ['저CLV', '중CLV', '고CLV'], summary
assert summary['nv_quadrants'] == ['저N·저V', '고N·저V', '저N·고V', '고N·고V'], summary
assert summary['high_clv_compositions'] == ['V우세 고CLV', '균형 고CLV', 'N우세 고CLV'], summary
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
assert summary['split'] == 'historical_development_days_684_690'
assert summary['fixed_clv_source'] == 'train-history N×V proxy at day 683'
print('불러온 모듈:', diagnostic_module.__file__)
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
report = run_fixed_segment_error_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1) 고정 CLV 구간별 M1 성과')
display(report['segment_metrics'])
print('2) 전체 N/V 4유형 구성')
display(report['nv_quadrant_population'])
print('3) 전체 N/V 4유형별 M1 성과')
display(report['nv_quadrant_metrics'])
print('4) 전체 N/V 4유형별 정답 누락 - Top-10 오추천 격차')
display(report['nv_quadrant_contrasts'])
print('5) 고CLV 내부 N/V 구성')
display(report['high_clv_composition_population'])
print('6) 고CLV 내부 N/V 구성별 M1 성과')
display(report['high_clv_composition_metrics'])
print('7) 고CLV 내부 정답 누락 - Top-10 오추천 격차')
display(report['high_clv_composition_contrasts'])
print('8) 고CLV 내부 실제 정답 누락·오추천 상품 예시')
display(report['high_clv_composition_examples'])
print('9) 기존 저·중·고 CLV 정답·적중·오추천 상품 특성')
display(report['item_role_summary'])
print('10) 기존 저·중·고 CLV 정답 누락상품 - Top-10 오추천상품 격차')
display(report['contrasts'])
print('11) 기존 저·중·고 CLV 구간·역할별 상위 카테고리')
display(report['category_summary'])
print('12) 기존 저·중·고 CLV 실제 정답 누락·오추천 상품 예시')
display(report['examples'])
print('저장 파일:', json.dumps(report['paths'], ensure_ascii=False, indent=2))